# diagrams for 2025 paper

# Circuit diagram

In [ ]:
from crsq_xp.circuits import time_evo_h1
from crsq.blocks.time_evolution import SUZUKI_TROTTER_ARITHMETIC, SUZUKI_TROTTER_QROM

delta_t = 0.001
bits = 5
num_nucl_iters = 1
num_elec_iters = 1
signed = True
parity = 'odd'
qn=1
use_saved_data = False
length = 16
psimax = 1.0
window_radius=16
x0=0

for st_method in [SUZUKI_TROTTER_ARITHMETIC, SUZUKI_TROTTER_QROM]:
    outdir = f"output/diagrams/{st_method}/{bits}bits"
    par = time_evo_h1.Driver1D(
        outdir,
        "CPU",
        False,
        "double",
        delta_t,
        bits,
        signed,
        parity,
        qn,
        length,
        psimax,
        window_radius,
        x0,
        num_elec_iters,
        num_nucl_iters,
        st_method,
        use_saved_data
    )

    par.draw_circuits()


# QC model の図

In [ ]:
from qiskit import QuantumCircuit
import math

th0 = 2*math.atan2(math.sqrt(7), math.sqrt(3))
th10 = 2*math.atan2(math.sqrt(2), math.sqrt(1))
th11 = 2*math.atan2(math.sqrt(4), math.sqrt(3))

qc = QuantumCircuit(2)
qc.ry(th0, 1)
qc.cry(th10, 1, 0, ctrl_state='0')
qc.cry(th11, 1, 0)
qc.draw('mpl', filename='output/diagrams/embed_example_circuit.png')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

# ===== 設定 =====
f_signal = 5.0      # 元の信号周波数
f_sample = 6.0      # サンプリング周波数（ナイキスト未満）

t_cont = np.linspace(0, 1, 2000)
t_sample = np.arange(0, 1, 1 / f_sample)

# ===== 元の信号 =====
x_cont = np.sin(2 * np.pi * f_signal * t_cont)
x_sample = np.sin(2 * np.pi * f_signal * t_sample)

# ===== エイリアス周波数 =====
n = round(f_signal / f_sample)
f_alias = abs(f_signal - n * f_sample)

# サンプル点を通るように位相を合わせる
phi = 2 * np.pi * (f_signal - f_alias) * t_sample[0]
x_alias = -np.sin(2 * np.pi * f_alias * t_cont + phi)

# ===== 描画 =====
plt.figure(figsize=(6, 6))

# 上段
plt.subplot(2, 1, 1)
plt.plot(t_cont, x_cont, label="Original signal")
plt.plot(t_sample, x_sample, 'o', label="Samples")
plt.title("Undersampling of a high-frequency signal")
plt.ylabel("Amplitude")
handles1, labels1 = plt.gca().get_legend_handles_labels()
#plt.legend()
plt.grid(True)

# 下段
plt.subplot(2, 1, 2)
plt.plot(t_cont, x_alias, '--', label=f"Aliased signal (f = {f_alias:.1f} Hz)")
plt.plot(t_sample, x_sample, 'o')
plt.title("Aliased low-frequency signal (passes sample points)")
plt.xlabel("Time")
plt.ylabel("Amplitude")
handles2, labels2 = plt.gca().get_legend_handles_labels()
plt.legend(handles1 + handles2, labels1 + labels2)
plt.grid(True)

plt.tight_layout()
#plt.show()
os.makedirs('output/diagrams', exist_ok=True)
plt.savefig('output/diagrams/aliasing_example.png')
